<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/01b_load_2wiki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
from pathlib import Path
from collections import Counter
from google.colab import drive

In [4]:
drive.mount("/content/drive")
WORKDIR = Path("/content/drive/MyDrive/HybridGraphRAG/twowiki")
WORKDIR.mkdir(parents=True, exist_ok=True)
print("Working folder: ", WORKDIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder:  /content/drive/MyDrive/HybridGraphRAG/twowiki


In [5]:
ZIP_URL = "https://www.dropbox.com/s/ms2m13252h6xubs/data_ids_april7.zip?dl=1"
zip_path = WORKDIR / "data_ids_april7.zip"

if not zip_path.exists():
    !wget -q -L -O "{zip_path}" "{ZIP_URL}"

!ls -lh "{WORKDIR}"

total 247M
-rw------- 1 root root 247M Oct 20 17:16 data_ids_april7.zip


In [8]:
import zipfile
with zipfile.ZipFile(zip_path) as z:
  print("files inside the zip:\n")
  for info in z.infolist():
    print(f"{info.file_size/1e6:9.1f} MB {info.filename}")

  dev_members = [n for n in z.namelist() if n.endswith("dev.json")]
  print("\nDev file(s) found: ",dev_members)

  if len(dev_members) != 1:
    raise RuntimeError("Expected exactly one dev file")

  with z.open(dev_members[0] )as src, open(WORKDIR / "dev.json","wb")as dst:
    dst.write(src.read())

print("\nExtracted to: ",WORKDIR/"dev.json")

files inside the zip:

     53.8 MB test.json
     57.6 MB dev.json
    707.8 MB train.json
     17.5 MB id_aliases.json

Dev file(s) found:  ['dev.json']

Extracted to:  /content/drive/MyDrive/HybridGraphRAG/twowiki/dev.json


In [12]:
with open(WORKDIR / "dev.json", "r", encoding ="utf-8") as f:
  data = json.load(f)

print("Total questions: ", len(data))
print("Key in one example: ", list(data[0].keys()))
print("Question types: ", Counter(d["type"] for d in data))

Total questions:  12576
Key in one example:  ['_id', 'type', 'question', 'context', 'entity_ids', 'supporting_facts', 'evidences', 'answer', 'evidences_id', 'answer_id']
Question types:  Counter({'compositional': 5236, 'comparison': 3040, 'bridge_comparison': 2751, 'inference': 1549})


In [13]:
ex = data[0]
for k,v in ex.items():
  print(f"\n----{k}----\n")
  print(json.dumps(v, ensure_ascii=False))


----_id----

"8813f87c0bdd11eba7f7acde48001122"

----type----

"compositional"

----question----

"Who is the mother of the director of film Polish-Russian War (Film)?"

----context----

[["Xawery Żuławski", ["Xawery Żuławski (born 22 December 1971 in Warsaw) is a Polish film director.", "In 1995 he graduated National Film School in Łódź.", "He is the son of actress Małgorzata Braunek and director Andrzej Żuławski.", "His second feature \"Wojna polsko-ruska\" (2009), adapted from the controversial best-selling novel by Dorota Masłowska, won First Prize in the New Polish Films competition at the 9th Era New Horizons Film Festival in Wrocław.", "In 2013, he stated he intends to direct a Polish novel \"Zły\" by Leopold Tyrmand.", "Żuławski and his wife Maria Strzelecka had 2 children together:", "son Kaj Żuławski (born 2002) and daughter Jagna Żuławska (born 2009)."]], ["Snow White and the Seven Dwarfs (1955 film)", ["Snow White and the Seven Dwarfs( USA:\" Snow White\") is a 1955 German

In [15]:
import random

SEED = 42
N_PER_TYPE = 5
rng = random.Random(SEED)

by_type = {}
for d in data:
  by_type.setdefault(d["type"], []).append(d)

sample = []
for qtype in sorted(by_type):
  sample.extend(rng.sample(by_type[qtype], N_PER_TYPE))

print("Sample size: ", len(sample))
print(Counter(d["type"] for d in sample), "\n")

for d in sample:
  print(f"[{d['type']}] {d['question']} --> {d['answer']}")

Sample size:  20
Counter({'bridge_comparison': 5, 'comparison': 5, 'compositional': 5, 'inference': 5}) 

[bridge_comparison] Which film has the director who was born first, Slunce, Seno, Jahody or To Sleep With Anger? --> To Sleep With Anger
[bridge_comparison] Which film has the director who died earlier, The Glass Menagerie (1987 Film) or Monsieur Sans-Gêne? --> Monsieur Sans-Gêne
[bridge_comparison] Which film whose director is younger, The Devil'S Pass Key or La Bella Di Lodi? --> La Bella Di Lodi
[bridge_comparison] Which film has the director who died earlier, Everything'S Ducky or Karthika (Film)? --> Everything'S Ducky
[bridge_comparison] Which film has the director died first, Monkey Shines or The Big Show-Off? --> The Big Show-Off
[comparison] Who was born earlier, Geoff Bagnall or Caroline Huppert? --> Caroline Huppert
[comparison] Are both Aslan, Tehran and Shell Lake, Wisconsin located in the same country? --> no
[comparison] Are the movies The Wings Of The Dove (1981 Fil

In [18]:
import re

def slugify(title):
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")[:40]

corpus = {}          # title -> chunk record
conflicts = 0

for d in sample:
    for title, sentences in d["context"]:
        if title in corpus:
            if corpus[title]["sentences"] != sentences:
                conflicts += 1
            continue
        idx = len(corpus)
        corpus[title] = {
            "id": f"{idx:04d}_{slugify(title)}",
            "text": f"{title}: " + " ".join(sentences),
            "source": title,
            "page": None,
            "chunk_index": 0,
            "title": title,
            "sentences": sentences,
        }

chunk_records = list(corpus.values())

questions = []
for d in sample:
    gold_titles = sorted({t for t, _ in d["supporting_facts"]})
    missing = [t for t in gold_titles if t not in corpus]
    assert not missing, f"Gold titles missing from corpus: {missing}"
    questions.append({
        "id": d["_id"],
        "type": d["type"],
        "question": d["question"],
        "answer": d["answer"],
        "answer_id": d["answer_id"],
        "gold_titles": gold_titles,
        "supporting_facts": d["supporting_facts"],
        "evidences": d["evidences"],
        "evidences_id": d["evidences_id"],
    })

print("Paragraphs in corpus:", len(chunk_records))
print("Duplicate-title text conflicts:", conflicts)
print("Questions:", len(questions))
print("\nExample chunk:\n", json.dumps(chunk_records[0], indent=2, ensure_ascii=False)[:700])

Paragraphs in corpus: 182
Duplicate-title text conflicts: 0
Questions: 20

Example chunk:
 {
  "id": "0000_caldecot_chubb",
  "text": "Caldecot Chubb: Caldecot\" Cotty\" Chubb( born March 3, 1950) is an American film producer who has produced films such as\" Eve's BayouHoffaUnthinkable The CrowDark Blue\" and\" Pootie Tang\". He won the Independent Spirit Award for Best First Feature in 1998 for\" Eve's Bayou\" and was a nominee for\" To Sleep with Anger\" in 1991. Both of those films have been admitted to the National Film Archive of the Library of Congress.",
  "source": "Caldecot Chubb",
  "page": null,
  "chunk_index": 0,
  "title": "Caldecot Chubb",
  "sentences": [
    "Caldecot\" Cotty\" Chubb( born March 3, 1950) is an American film producer who has produced films such as\


In [19]:
chunks_out = WORKDIR / "document_chunks.jsonl"
questions_out = WORKDIR / "questions.jsonl"

for path, records in [(chunks_out, chunk_records), (questions_out, questions)]:
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("Saved", len(records), "records to", path)

Saved 182 records to /content/drive/MyDrive/HybridGraphRAG/twowiki/document_chunks.jsonl
Saved 20 records to /content/drive/MyDrive/HybridGraphRAG/twowiki/questions.jsonl
